In [0]:
spark.table("workspace.logistics_project.customers").printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- credit_terms_days: long (nullable = true)
 |-- primary_freight_type: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- contract_start_date: date (nullable = true)
 |-- annual_revenue_potential: long (nullable = true)



In [0]:
spark.table("workspace.logistics_project.loads").printSchema()

root
 |-- load_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- weight_lbs: long (nullable = true)
 |-- pieces: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- fuel_surcharge: double (nullable = true)
 |-- accessorial_charges: long (nullable = true)
 |-- load_status: string (nullable = true)
 |-- booking_type: string (nullable = true)



In [0]:
#Read Tables
import pyspark.sql.functions as F

customers = spark.table("workspace.logistics_project.customers")
loads = spark.table("workspace.logistics_project.loads")

In [0]:
#Join Tables
customer_analysis = (
    loads
    .join(customers, "customer_id", "inner")
)

In [0]:
#Calculate Customer KPIs
customer_metrics = (
    customer_analysis
    .groupBy(
        "customer_id",
        "customer_name",
        "customer_type",
        "account_status",
        "primary_freight_type"
    )
    .agg(
        F.count("load_id").alias("total_loads"),
        F.round(F.sum("revenue"),2).alias("total_revenue"),
        F.sum("weight_lbs").alias("total_weight_lbs"),
        F.sum("pieces").alias("total_pieces"),
        F.round(F.avg("revenue"),2).alias("avg_revenue_per_load")
    )
)

In [0]:
#Revenue Contribution %
total_revenue = customer_metrics.agg(
    F.sum("total_revenue")
).collect()[0][0]

customer_metrics = customer_metrics.withColumn(
    "revenue_contribution_pct",
    F.round(
        (F.col("total_revenue") / F.lit(total_revenue)) * 100,
        2
    )
)

In [0]:
#Revenue Per Pound
customer_metrics = customer_metrics.withColumn(
    "revenue_per_lb",
    F.round(
        F.col("total_revenue") /
        F.col("total_weight_lbs"),
        4
    )
)

In [0]:
#Display Results
display(
    customer_metrics.orderBy(
        F.col("total_revenue").desc()
    )
)

customer_id,customer_name,customer_type,account_status,primary_freight_type,total_loads,total_revenue,total_weight_lbs,total_pieces,avg_revenue_per_load,revenue_contribution_pct,revenue_per_lb
CUST00200,XYZ Foods,Dedicated,Inactive,Retail,476,1544419.81,13311382,6783,3244.58,0.59,0.116
CUST00181,Superior Group,Contract,Active,Automotive,497,1542321.02,13660059,7002,3103.26,0.59,0.1129
CUST00077,Metro Group,Spot,Inactive,Retail,487,1521982.07,12851362,7306,3125.22,0.58,0.1184
CUST00097,National Wholesale,Contract,Active,Retail,470,1487129.31,12831529,6488,3164.1,0.57,0.1159
CUST00122,Metro Foods,Dedicated,Active,Automotive,463,1483188.9,12649238,6755,3203.43,0.56,0.1173
CUST00028,First Group,Contract,Active,Automotive,476,1481527.84,13223524,6794,3112.45,0.56,0.112
CUST00110,Continental Group,Spot,Active,Electronics,481,1479584.73,12904432,6860,3076.06,0.56,0.1147
CUST00101,United Corp,Contract,Active,Automotive,460,1477854.32,12688285,6498,3212.73,0.56,0.1165
CUST00124,First Supply Chain,Contract,Active,Food/Beverage,454,1472131.31,12317167,6792,3242.58,0.56,0.1195
CUST00196,XYZ Logistics,Spot,Active,Retail,483,1471132.9,13125712,6835,3045.82,0.56,0.1121


In [0]:
display(customer_metrics)

customer_id,customer_name,customer_type,account_status,primary_freight_type,total_loads,total_revenue,total_weight_lbs,total_pieces,avg_revenue_per_load,revenue_contribution_pct,revenue_per_lb
CUST00075,Elite Corp,Contract,Inactive,General,445,1390501.92,12231785,6447,3124.72,0.53,0.1137
CUST00029,Superior Distribution,Dedicated,Active,Food/Beverage,450,1365203.09,12508957,6551,3033.78,0.52,0.1091
CUST00135,Premier Retail,Dedicated,Inactive,Retail,465,1437776.93,12868006,6635,3091.99,0.55,0.1117
CUST00190,Global Wholesale,Dedicated,Active,Food/Beverage,448,1370567.97,12297337,6677,3059.3,0.52,0.1115
CUST00116,Premier Logistics,Spot,Active,Food/Beverage,419,1288735.66,11341987,5933,3075.74,0.49,0.1136
CUST00078,Premier Wholesale,Spot,Inactive,Retail,424,1307620.36,11292161,6056,3084.01,0.5,0.1158
CUST00012,ABC Distribution,Dedicated,Active,Electronics,453,1374502.5,12208752,6813,3034.22,0.52,0.1126
CUST00182,XYZ Foods,Spot,Active,Consumer Goods,415,1338825.87,11469520,6236,3226.09,0.51,0.1167
CUST00107,Premier Industries,Dedicated,Active,Food/Beverage,437,1377741.38,12014366,6303,3152.73,0.52,0.1147
CUST00171,Premier Wholesale,Contract,Active,Automotive,403,1265181.02,11183146,5800,3139.41,0.48,0.1131


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#Create Gold Table
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+--------------------+-----------+
|database      |tableName           |isTemporary|
+--------------+--------------------+-----------+
|logistics_gold|driver_performance  |false      |
|logistics_gold|fleet_utilization   |false      |
|logistics_gold|fuel_efficiency     |false      |
|logistics_gold|maintenance_analysis|false      |
|logistics_gold|route_profitability |false      |
+--------------+--------------------+-----------+



In [0]:
customer_metrics.write \
.format("delta") \
.saveAsTable(
    "workspace.logistics_gold.customer_analysis"
)

In [0]:
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+--------------------+-----------+
|database      |tableName           |isTemporary|
+--------------+--------------------+-----------+
|logistics_gold|customer_analysis   |false      |
|logistics_gold|driver_performance  |false      |
|logistics_gold|fleet_utilization   |false      |
|logistics_gold|fuel_efficiency     |false      |
|logistics_gold|maintenance_analysis|false      |
|logistics_gold|route_profitability |false      |
+--------------+--------------------+-----------+



In [0]:
#MERGE
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.customer_analysis"
)

gold_table.alias("target").merge(
    customer_metrics.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#verify
display(
    spark.table(
        "workspace.logistics_gold.customer_analysis"
    )
)

customer_id,customer_name,customer_type,account_status,primary_freight_type,total_loads,total_revenue,total_weight_lbs,total_pieces,avg_revenue_per_load,revenue_contribution_pct,revenue_per_lb
CUST00075,Elite Corp,Contract,Inactive,General,445,1390501.92,12231785,6447,3124.72,0.53,0.1137
CUST00029,Superior Distribution,Dedicated,Active,Food/Beverage,450,1365203.09,12508957,6551,3033.78,0.52,0.1091
CUST00135,Premier Retail,Dedicated,Inactive,Retail,465,1437776.93,12868006,6635,3091.99,0.55,0.1117
CUST00190,Global Wholesale,Dedicated,Active,Food/Beverage,448,1370567.97,12297337,6677,3059.3,0.52,0.1115
CUST00116,Premier Logistics,Spot,Active,Food/Beverage,419,1288735.66,11341987,5933,3075.74,0.49,0.1136
CUST00078,Premier Wholesale,Spot,Inactive,Retail,424,1307620.36,11292161,6056,3084.01,0.5,0.1158
CUST00012,ABC Distribution,Dedicated,Active,Electronics,453,1374502.5,12208752,6813,3034.22,0.52,0.1126
CUST00182,XYZ Foods,Spot,Active,Consumer Goods,415,1338825.87,11469520,6236,3226.09,0.51,0.1167
CUST00107,Premier Industries,Dedicated,Active,Food/Beverage,437,1377741.38,12014366,6303,3152.73,0.52,0.1147
CUST00171,Premier Wholesale,Contract,Active,Automotive,403,1265181.02,11183146,5800,3139.41,0.48,0.1131
